# Tiny Transformer for Tiny Shakespeare

This notebook implements a lightweight decoder-only Transformer in PyTorch for next-token prediction on the Tiny Shakespeare corpus. The workflow follows the assignment directly: subword tokenization, overlapping sequence construction, manual positional encoding, RMSNorm, causal self-attention, training with cross-entropy loss, validation perplexity, loss curves, and attention heatmaps.

## 1. Title + Assignment Overview

The goal of this assignment is to build and train a small Transformer language model from scratch for next-token prediction. In addition to training the model, this notebook includes attention visualization and validation perplexity so the results can directly support the final written report.

## 2. Imports and Reproducibility

Reproducibility matters because Transformer training is sensitive to initialization, batching, and randomization. This section imports the required libraries, defines the experiment configuration, and seeds Python, NumPy, and PyTorch so the run is easy to reproduce.

In [ ]:

from dataclasses import dataclass
from pathlib import Path
import math
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
from torch.utils.data import DataLoader, Dataset


PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()


def apply_plot_style() -> None:
    plt.rcParams.update(
        {
            "figure.figsize": (8, 4.5),
            "axes.titlesize": 14,
            "axes.labelsize": 12,
            "xtick.labelsize": 10,
            "ytick.labelsize": 10,
            "legend.fontsize": 10,
            "figure.dpi": 120,
        }
    )


def format_token_labels(
    token_labels: list[str],
    max_tokens: int | None = None,
    max_label_length: int = 10,
) -> list[str]:
    labels = token_labels[:max_tokens] if max_tokens is not None else token_labels
    formatted = []
    for label in labels:
        clean = label.replace(chr(10), "\n").strip() or "[SPACE]"
        if len(clean) > max_label_length:
            clean = clean[: max(1, max_label_length - 3)] + "..."
        formatted.append(clean)
    return formatted


def plot_metric_curve(
    values: list[float],
    save_path: str | Path,
    title: str,
    ylabel: str,
    xlabel: str = "Epoch",
    marker: str = "o",
    color: str = "#1f77b4",
) -> None:
    apply_plot_style()
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    epochs = np.arange(1, len(values) + 1)

    fig, ax = plt.subplots()
    ax.plot(epochs, values, marker=marker, linewidth=2, color=color)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


def plot_loss_curves(
    train_loss: list[float],
    val_loss: list[float],
    save_path: str | Path,
    title: str = "Tiny Transformer Training and Validation Loss",
) -> None:
    apply_plot_style()
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    epochs = np.arange(1, len(train_loss) + 1)

    fig, ax = plt.subplots()
    ax.plot(epochs, train_loss, marker="o", linewidth=2, label="Training loss", color="#1f77b4")
    ax.plot(epochs, val_loss, marker="s", linewidth=2, label="Validation loss", color="#d62728")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-entropy loss")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


def plot_runtime_summary(
    epoch_seconds: list[float],
    peak_memory_mb: list[float],
    save_path: str | Path,
) -> None:
    apply_plot_style()
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    epochs = np.arange(1, len(epoch_seconds) + 1)
    show_memory = any(value > 0 for value in peak_memory_mb)

    if show_memory:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        runtime_ax, memory_ax = axes
    else:
        fig, runtime_ax = plt.subplots(figsize=(8, 4.5))
        memory_ax = None

    runtime_ax.bar(epochs, epoch_seconds, color="#2ca02c", alpha=0.85)
    runtime_ax.set_title("Runtime Per Epoch")
    runtime_ax.set_xlabel("Epoch")
    runtime_ax.set_ylabel("Seconds")
    runtime_ax.grid(axis="y", alpha=0.3)

    if memory_ax is not None:
        memory_ax.bar(epochs, peak_memory_mb, color="#9467bd", alpha=0.85)
        memory_ax.set_title("Peak GPU Memory Per Epoch")
        memory_ax.set_xlabel("Epoch")
        memory_ax.set_ylabel("MB")
        memory_ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


def plot_attention_heatmap(
    attention_map: torch.Tensor,
    token_labels: list[str],
    save_path: str | Path,
    title: str,
) -> None:
    apply_plot_style()
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    matrix = np.asarray(attention_map.detach().cpu())
    labels = format_token_labels(token_labels, max_tokens=matrix.shape[0], max_label_length=9)

    fig, ax = plt.subplots(figsize=(8, 6))
    image = ax.imshow(matrix, cmap="magma", aspect="auto")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="Attention weight")
    ax.set_title(title)
    ax.set_xlabel("Key / attended token")
    ax.set_ylabel("Query / current token")
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=90)
    ax.set_yticklabels(labels)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


def plot_attention_grid(
    attention_tensor: torch.Tensor,
    token_labels: list[str],
    head_indices: list[int],
    save_path: str | Path,
    title: str,
) -> None:
    apply_plot_style()
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    labels = format_token_labels(token_labels, max_tokens=attention_tensor.shape[-1], max_label_length=8)
    num_heads = len(head_indices)
    cols = min(2, num_heads)
    rows = int(np.ceil(num_heads / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.8 * rows))
    axes = np.atleast_1d(axes).ravel()

    for axis, head_idx in zip(axes, head_indices):
        matrix = np.asarray(attention_tensor[0, head_idx].detach().cpu())
        image = axis.imshow(matrix, cmap="magma", aspect="auto")
        axis.set_title(f"Head {head_idx + 1}")
        axis.set_xlabel("Key")
        axis.set_ylabel("Query")
        axis.set_xticks(np.arange(len(labels)))
        axis.set_yticks(np.arange(len(labels)))
        axis.set_xticklabels(labels, rotation=90)
        axis.set_yticklabels(labels)
        fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)

    for axis in axes[num_heads:]:
        axis.axis("off")

    fig.suptitle(title, fontsize=15)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


def plot_attention_comparison(
    left_attention: torch.Tensor,
    right_attention: torch.Tensor,
    token_labels: list[str],
    save_path: str | Path,
    left_title: str,
    right_title: str,
    figure_title: str,
) -> None:
    apply_plot_style()
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    left_matrix = np.asarray(left_attention.detach().cpu())
    right_matrix = np.asarray(right_attention.detach().cpu())
    labels = format_token_labels(token_labels, max_tokens=left_matrix.shape[0], max_label_length=8)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    for axis, matrix, subtitle in zip(
        axes,
        [left_matrix, right_matrix],
        [left_title, right_title],
    ):
        image = axis.imshow(matrix, cmap="magma", aspect="auto")
        axis.set_title(subtitle)
        axis.set_xlabel("Key")
        axis.set_ylabel("Query")
        axis.set_xticks(np.arange(len(labels)))
        axis.set_yticks(np.arange(len(labels)))
        axis.set_xticklabels(labels, rotation=90)
        axis.set_yticklabels(labels)
        fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)

    fig.suptitle(figure_title, fontsize=15)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


def plot_experiment_metric_comparison(
    metric_history_by_label: dict[str, list[float]],
    save_path: str | Path,
    title: str,
    ylabel: str,
    xlabel: str = "Epoch",
) -> None:
    apply_plot_style()
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots()
    for label, values in metric_history_by_label.items():
        epochs = np.arange(1, len(values) + 1)
        ax.plot(epochs, values, marker="o", linewidth=2, label=label)

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


@dataclass
class ExperimentConfig:
    """Central configuration for the end-to-end experiment."""

    seed: int = 42
    vocab_size: int = 500
    context_length: int = 50
    stride: int = 2
    batch_size: int = 64
    embedding_dim: int = 128
    num_heads: int = 4
    num_layers: int = 2
    ff_multiplier: int = 4
    dropout: float = 0.1
    learning_rate: float = 3e-4
    weight_decay: float = 1e-2
    epochs: int = 6
    max_generation_tokens: int = 80


def seed_everything(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch for reproducible runs."""

    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    """Return the best available compute device."""

    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def count_parameters(model: torch.nn.Module) -> int:
    """Count trainable model parameters."""

    return sum(param.numel() for param in model.parameters() if param.requires_grad)


config = ExperimentConfig()
seed_everything(config.seed)
device = get_device()

DATA_PATH = PROJECT_ROOT / "data" / "tiny_shakespeare.txt"
TOKENIZER_PATH = PROJECT_ROOT / "data" / "tiny_shakespeare_bpe.json"
FIGURES_DIR = PROJECT_ROOT / "figures"

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device}")
print(config)


## 3. Data Loading

The Tiny Shakespeare corpus is a compact text dataset that is commonly used for language-model experiments. The task here is next-token prediction: given a sequence of tokens, the model learns to predict the token that should come next at every position.

In [ ]:
def load_text_file(path: str | Path) -> str:
    """Load the Tiny Shakespeare corpus from disk."""

    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Expected dataset at {path}. Download Tiny Shakespeare and place it there."
        )
    return path.read_text(encoding="utf-8")


text = load_text_file(DATA_PATH)
print(f"Number of characters: {len(text):,}")
print(text[:500])

## 4. Tokenization

Instead of character-level tokenization, this notebook uses a small subword tokenizer based on Byte Pair Encoding (BPE). Subword tokenization is useful because it captures reusable text fragments such as prefixes, suffixes, and common word pieces, which is more expressive than single characters while still keeping the vocabulary compact. The vocabulary size is capped at 500 to satisfy the assignment constraint and keep training lightweight.

In [ ]:
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]


def train_or_load_tokenizer(
    text_path: str | Path,
    tokenizer_path: str | Path,
    vocab_size: int = 500,
) -> Tokenizer:
    """Train a compact BPE tokenizer once, then reuse the saved JSON."""

    text_path = Path(text_path)
    tokenizer_path = Path(tokenizer_path)
    tokenizer_path.parent.mkdir(parents=True, exist_ok=True)

    if tokenizer_path.exists():
        return Tokenizer.from_file(str(tokenizer_path))

    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()
    tokenizer.decoder = decoders.Metaspace()

    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=SPECIAL_TOKENS,
        show_progress=False,
    )
    tokenizer.train([str(text_path)], trainer=trainer)
    tokenizer.save(str(tokenizer_path))
    return tokenizer


tokenizer = train_or_load_tokenizer(
    text_path=DATA_PATH,
    tokenizer_path=TOKENIZER_PATH,
    vocab_size=config.vocab_size,
)

encoded = tokenizer.encode(text)
token_ids = encoded.ids
print(f"Vocabulary size: {tokenizer.get_vocab_size()}")
print(f"Number of tokens in corpus: {len(token_ids):,}")
print("First 25 tokens:", encoded.tokens[:25])

## 5. Sequence Construction

Transformers train on fixed-length windows. To turn the tokenized corpus into training data, we create overlapping sequences of a chosen context length. For each window, the input is the first N tokens and the target is the same sequence shifted one step to the left, which exactly matches the next-token prediction objective.

In [ ]:
def build_input_target_sequences(
    token_ids: list[int],
    context_length: int,
    stride: int = 1,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Create overlapping next-token prediction windows."""

    if context_length <= 0:
        raise ValueError("context_length must be positive.")
    if stride <= 0:
        raise ValueError("stride must be positive.")
    if len(token_ids) <= context_length:
        raise ValueError("Tokenized corpus is too short for the chosen context length.")

    inputs = []
    targets = []

    for start_idx in range(0, len(token_ids) - context_length, stride):
        end_idx = start_idx + context_length
        inputs.append(token_ids[start_idx:end_idx])
        # Targets are the same window shifted by one token.
        targets.append(token_ids[start_idx + 1 : end_idx + 1])

    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)


inputs, targets = build_input_target_sequences(
    token_ids=token_ids,
    context_length=config.context_length,
    stride=config.stride,
)

print("Input tensor shape :", tuple(inputs.shape))
print("Target tensor shape:", tuple(targets.shape))

sample_index = 0
sample_input_ids = inputs[sample_index].tolist()
sample_target_ids = targets[sample_index].tolist()
print("Sample input ids :", sample_input_ids[:12], "...")
print("Sample target ids:", sample_target_ids[:12], "...")
print("Decoded input :", tokenizer.decode(sample_input_ids))
print("Decoded target:", tokenizer.decode(sample_target_ids))

## 6. Train/Validation Split

The assignment asks for an 80/20 train-validation split. A separate test set is not necessary here because the goal is to learn the architecture, check whether training is stable, and report validation perplexity as the main evaluation metric.

In [ ]:
def train_validation_split(
    inputs: torch.Tensor,
    targets: torch.Tensor,
    train_fraction: float = 0.8,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Split aligned input-target tensors while preserving sequence order."""

    if not 0.0 < train_fraction < 1.0:
        raise ValueError("train_fraction must be between 0 and 1.")

    split_idx = int(train_fraction * len(inputs))
    return (
        inputs[:split_idx],
        targets[:split_idx],
        inputs[split_idx:],
        targets[split_idx:],
    )


train_inputs, train_targets, val_inputs, val_targets = train_validation_split(
    inputs,
    targets,
    train_fraction=0.8,
)

print(f"Training sequences  : {len(train_inputs):,}")
print(f"Validation sequences: {len(val_inputs):,}")

## 7. Dataloaders

PyTorch Dataset and DataLoader objects make minibatch training simple and memory-efficient. The batch size stays moderate so the experiment can run on a normal laptop while still processing the corpus in parallel.

In [ ]:
class TokenSequenceDataset(Dataset):
    """Simple dataset wrapper for aligned token windows and targets."""

    def __init__(self, inputs: torch.Tensor, targets: torch.Tensor) -> None:
        if len(inputs) != len(targets):
            raise ValueError("Inputs and targets must have the same length.")
        self.inputs = inputs
        self.targets = targets

    def __len__(self) -> int:
        return len(self.inputs)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.inputs[idx], self.targets[idx]


def create_dataloaders(
    train_inputs: torch.Tensor,
    train_targets: torch.Tensor,
    val_inputs: torch.Tensor,
    val_targets: torch.Tensor,
    batch_size: int,
) -> tuple[DataLoader, DataLoader]:
    """Build DataLoaders for training and validation."""

    if batch_size <= 0:
        raise ValueError("batch_size must be positive.")

    train_dataset = TokenSequenceDataset(train_inputs, train_targets)
    val_dataset = TokenSequenceDataset(val_inputs, val_targets)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader


train_loader, val_loader = create_dataloaders(
    train_inputs=train_inputs,
    train_targets=train_targets,
    val_inputs=val_inputs,
    val_targets=val_targets,
    batch_size=config.batch_size,
)

example_batch_inputs, example_batch_targets = next(iter(train_loader))
print("Batch input shape :", tuple(example_batch_inputs.shape))
print("Batch target shape:", tuple(example_batch_targets.shape))

## 8. Positional Encoding

Self-attention alone has no built-in notion of word order, so positional encoding is necessary to tell the model where each token appears in the sequence. This implementation uses a manual sinusoidal positional encoding because it is simple, deterministic, and directly matches one of the assignment options.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    """Add fixed sinusoidal position information to token embeddings."""

    def __init__(self, d_model: int, max_length: int = 512) -> None:
        super().__init__()

        positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)
        div_terms = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_length, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(positions * div_terms)
        pe[:, 1::2] = torch.cos(positions * div_terms)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]


positional_encoder = SinusoidalPositionalEncoding(config.embedding_dim, max_length=config.context_length)
dummy_embeddings = torch.zeros(1, config.context_length, config.embedding_dim)
encoded_positions = positional_encoder(dummy_embeddings)
print("Positional encoding shape:", tuple(encoded_positions.shape))
print(encoded_positions[0, :3, :8])

## 9. RMSNorm

Normalization stabilizes training by controlling the scale of hidden states. This notebook uses RMSNorm, which rescales activations using their root-mean-square value without subtracting the mean, keeping the implementation lightweight and aligned with the assignment requirements.

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square normalization without mean-centering."""

    def __init__(self, dim: int, eps: float = 1e-8) -> None:
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return self.scale * (x / rms)


rmsnorm = RMSNorm(config.embedding_dim)
normalized = rmsnorm(torch.randn(2, config.context_length, config.embedding_dim))
print("RMSNorm output shape:", tuple(normalized.shape))

## 10. Self-Attention with Causal Mask

The core Transformer operation is scaled dot-product self-attention. For language modeling, the model must not look ahead at future tokens, so a causal mask is applied to block attention to positions on the right. The implementation stores attention weights so they can be visualized later.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """Manual multi-head self-attention with a causal mask."""

    def __init__(self, d_model: int, num_heads: int, dropout: float) -> None:
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads.")

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor, return_attention: bool = False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        batch_size, seq_len, _ = x.shape

        q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        attention_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # The causal mask blocks access to future tokens during autoregressive training.
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool), diagonal=1
        )
        attention_scores = attention_scores.masked_fill(causal_mask, float("-inf"))

        attention_weights = F.softmax(attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        attended = torch.matmul(attention_weights, v)
        attended = attended.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.out_proj(attended)

        if return_attention:
            return output, attention_weights.detach()
        return output, None


attention_layer = MultiHeadSelfAttention(
    d_model=config.embedding_dim,
    num_heads=config.num_heads,
    dropout=config.dropout,
)
dummy_sequence = torch.randn(2, config.context_length, config.embedding_dim)
attention_output, attention_weights = attention_layer(dummy_sequence, return_attention=True)

print("Attention output shape:", tuple(attention_output.shape))
print("Attention weights shape:", tuple(attention_weights.shape))

## 11. Feed-Forward Network

Each Transformer block also includes a position-wise feed-forward network. This sublayer applies the same nonlinear transformation at every position and gives the model extra capacity beyond the attention mechanism itself.

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network used inside each Transformer block."""

    def __init__(self, d_model: int, ff_multiplier: int, dropout: float) -> None:
        super().__init__()
        hidden_dim = ff_multiplier * d_model
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


feed_forward = FeedForward(
    d_model=config.embedding_dim,
    ff_multiplier=config.ff_multiplier,
    dropout=config.dropout,
)
ffn_output = feed_forward(dummy_sequence)

print("Feed-forward output shape:", tuple(ffn_output.shape))

## 12. Transformer Block

A Transformer block combines RMSNorm, self-attention, a residual connection, another RMSNorm, a feed-forward network, and a second residual connection. Organizing the architecture this way makes the flow of information easy to inspect and mirrors the standard decoder-only design used for autoregressive language models.

In [ ]:
class TransformerBlock(nn.Module):
    """Pre-norm decoder block with attention, FFN, and residual connections."""

    def __init__(self, d_model: int, num_heads: int, ff_multiplier: int, dropout: float) -> None:
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = FeedForward(d_model, ff_multiplier, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor, return_attention: bool = False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        attn_output, attention_weights = self.attn(self.attn_norm(x), return_attention=return_attention)
        x = x + self.dropout(attn_output)
        x = x + self.dropout(self.ffn(self.ffn_norm(x)))
        return x, attention_weights


transformer_block = TransformerBlock(
    d_model=config.embedding_dim,
    num_heads=config.num_heads,
    ff_multiplier=config.ff_multiplier,
    dropout=config.dropout,
)
block_output, block_attention = transformer_block(dummy_sequence, return_attention=True)

print("Transformer block output shape:", tuple(block_output.shape))
print("Transformer block attention shape:", tuple(block_attention.shape))

## 13. Tiny Transformer Language Model

The full model stacks a small number of Transformer blocks on top of token embeddings and sinusoidal positional encodings, then maps hidden states to vocabulary logits with a final linear layer. The default architecture uses 2 blocks and an embedding dimension of 128 to stay lightweight, as suggested in the assignment.

In [ ]:
class TinyTransformerLM(nn.Module):
    """Small decoder-only Transformer for next-token prediction."""

    def __init__(
        self,
        vocab_size: int,
        context_length: int,
        d_model: int = 128,
        num_heads: int = 4,
        num_layers: int = 2,
        ff_multiplier: int = 4,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_encoding = SinusoidalPositionalEncoding(d_model, max_length=context_length)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [
                TransformerBlock(d_model, num_heads, ff_multiplier, dropout)
                for _ in range(num_layers)
            ]
        )
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(
        self,
        input_ids: torch.Tensor,
        targets: torch.Tensor | None = None,
        return_attentions: bool = False,
    ) -> tuple[torch.Tensor, torch.Tensor | None, list[torch.Tensor]]:
        if input_ids.size(1) > self.context_length:
            raise ValueError("Input sequence is longer than the configured context length.")

        x = self.token_embedding(input_ids)
        x = self.position_encoding(x)
        x = self.dropout(x)

        attentions = []
        for block in self.blocks:
            x, attn_map = block(x, return_attention=return_attentions)
            if attn_map is not None:
                attentions.append(attn_map)

        logits = self.lm_head(self.final_norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, attentions

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
    ) -> torch.Tensor:
        """Autoregressively sample tokens from the model."""

        self.eval()
        generated = input_ids

        for _ in range(max_new_tokens):
            # Only the most recent context window is fed back into the model.
            context = generated[:, -self.context_length :]
            logits, _, _ = self(context)
            next_token_logits = logits[:, -1, :] / max(temperature, 1e-6)
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            generated = torch.cat([generated, next_token], dim=1)

        return generated


model = TinyTransformerLM(
    vocab_size=tokenizer.get_vocab_size(),
    context_length=config.context_length,
    d_model=config.embedding_dim,
    num_heads=config.num_heads,
    num_layers=config.num_layers,
    ff_multiplier=config.ff_multiplier,
    dropout=config.dropout,
).to(device)

print(model)
print(f"Trainable parameters: {count_parameters(model):,}")

## 14. Training Setup

Next-token prediction is a multi-class classification problem at every position in the sequence, so cross-entropy loss is the correct training objective. AdamW is used as the optimizer because it works well for small Transformer models and handles weight decay cleanly.

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)
print(optimizer)

## 15. Training Loop

The training loop runs for a modest number of epochs and records both training and validation loss after each pass through the data. These values will later be plotted to show how the model learns and whether overfitting appears.

In [ ]:
def clone_state_dict_to_cpu(model: torch.nn.Module) -> dict[str, torch.Tensor]:
    """Create a CPU copy of a model state dict for lightweight checkpointing."""

    return {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}


def train_one_epoch(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    """Run one full training epoch and return the mean loss."""

    model.train()
    running_loss = 0.0

    for inputs, targets in dataloader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        _, loss, _ = model(inputs, targets=targets)
        assert loss is not None
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / max(len(dataloader), 1)


@torch.no_grad()
def evaluate_loss(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    device: torch.device,
) -> float:
    """Evaluate mean loss on a validation dataloader."""

    model.eval()
    running_loss = 0.0

    for inputs, targets in dataloader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        _, loss, _ = model(inputs, targets=targets)
        assert loss is not None
        running_loss += loss.item()

    return running_loss / max(len(dataloader), 1)


def fit(
    model: torch.nn.Module,
    train_loader: torch.utils.data.DataLoader,
    val_loader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    epochs: int,
) -> dict[str, list[float] | dict[str, dict[str, torch.Tensor]]]:
    """Train the model and store report-friendly metrics for every epoch."""

    history: dict[str, list[float] | dict[str, dict[str, torch.Tensor]]] = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_perplexity": [],
        "epoch_seconds": [],
        "peak_memory_mb": [],
        "checkpoints": {},
    }

    for epoch in range(1, epochs + 1):
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(device)

        start_time = time.perf_counter()
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_loss = evaluate_loss(model, val_loader, device)
        epoch_seconds = time.perf_counter() - start_time
        val_perplexity = math.exp(val_loss)
        peak_memory_mb = (
            torch.cuda.max_memory_allocated(device) / (1024 ** 2)
            if device.type == "cuda"
            else 0.0
        )

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_perplexity"].append(val_perplexity)
        history["epoch_seconds"].append(epoch_seconds)
        history["peak_memory_mb"].append(peak_memory_mb)

        if epoch == 1:
            history["checkpoints"]["epoch_1"] = clone_state_dict_to_cpu(model)

        print(
            f"Epoch {epoch:02d}/{epochs:02d} | "
            f"train loss: {train_loss:.4f} | "
            f"val loss: {val_loss:.4f} | "
            f"val ppl: {val_perplexity:.2f} | "
            f"time: {epoch_seconds:.2f}s"
        )

    return history


history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=config.epochs,
)
history


## 16. Validation Perplexity

Perplexity is a standard metric for language modeling. It is defined here as the exponential of the validation cross-entropy loss, so lower perplexity means the model is assigning higher probability to the correct next tokens on held-out validation sequences.

In [ ]:
def loss_to_perplexity(loss: float) -> float:
    """Convert cross-entropy loss to perplexity."""

    return math.exp(loss)


@torch.no_grad()
def compute_validation_perplexity(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    device: torch.device,
) -> tuple[float, float]:
    """Return validation loss together with perplexity."""

    model.eval()
    total_loss = 0.0

    for inputs, targets in dataloader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        _, loss, _ = model(inputs, targets=targets)
        assert loss is not None
        total_loss += loss.item()

    mean_loss = total_loss / max(len(dataloader), 1)
    return mean_loss, loss_to_perplexity(mean_loss)


val_loss, val_perplexity = compute_validation_perplexity(model, val_loader, device)

summary_lines = [
    f"Trainable parameters : {count_parameters(model):,}",
    f"Epochs completed     : {len(history['epoch'])}",
    f"Final training loss  : {history['train_loss'][-1]:.4f}",
    f"Final validation loss: {val_loss:.4f}",
    f"Final val perplexity : {val_perplexity:.4f}",
    f"Average epoch time   : {np.mean(history['epoch_seconds']):.2f} seconds",
]
if device.type == "cuda":
    summary_lines.append(
        f"Peak GPU memory      : {max(history['peak_memory_mb']):.1f} MB"
    )

print("\n".join(summary_lines))

summary_path = FIGURES_DIR / "training_summary.txt"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
print(f"Saved training summary to: {summary_path}")


## 17. Loss, Perplexity, and Runtime Visualization

These figures summarize optimization behavior over time. Separate training and validation curves make it easier to discuss stability and generalization, the combined curve is the main report figure, and the perplexity plot converts validation loss into the assignment's primary evaluation metric. Runtime and memory summaries support the discussion of computational bottlenecks.


In [ ]:
train_loss_path = FIGURES_DIR / "train_loss.png"
plot_metric_curve(
    values=history["train_loss"],
    save_path=train_loss_path,
    title="Training Loss Over Epochs",
    ylabel="Cross-entropy loss",
    color="#1f77b4",
)

val_loss_path = FIGURES_DIR / "val_loss.png"
plot_metric_curve(
    values=history["val_loss"],
    save_path=val_loss_path,
    title="Validation Loss Over Epochs",
    ylabel="Cross-entropy loss",
    color="#d62728",
)

loss_curve_path = FIGURES_DIR / "loss_curves.png"
plot_loss_curves(
    train_loss=history["train_loss"],
    val_loss=history["val_loss"],
    save_path=loss_curve_path,
)

perplexity_curve_path = FIGURES_DIR / "perplexity_curve.png"
plot_metric_curve(
    values=history["val_perplexity"],
    save_path=perplexity_curve_path,
    title="Validation Perplexity Over Epochs",
    ylabel="Perplexity",
    color="#ff7f0e",
)

runtime_summary_path = FIGURES_DIR / "runtime_summary.png"
plot_runtime_summary(
    epoch_seconds=history["epoch_seconds"],
    peak_memory_mb=history["peak_memory_mb"],
    save_path=runtime_summary_path,
)

print(f"Saved training loss curve to: {train_loss_path}")
print(f"Saved validation loss curve to: {val_loss_path}")
print(f"Saved combined loss curve to: {loss_curve_path}")
print(f"Saved validation perplexity curve to: {perplexity_curve_path}")
print(f"Saved runtime summary figure to: {runtime_summary_path}")


## 18. Attention Visualization Suite

Attention is the most important interpretability component in this assignment, so the notebook now saves several complementary views instead of a single heatmap. The figures below show one clean example, compare heads within a layer, compare layers on the same sequence, and compare an early checkpoint against the final model to show how attention sharpens during training.


In [ ]:
def token_ids_to_labels(tokenizer: Tokenizer, token_ids: list[int]) -> list[str]:
    """Map token ids back to readable token strings for visualization."""

    return [tokenizer.id_to_token(token_id) or "[UNK]" for token_id in token_ids]


attention_token_count = min(18, config.context_length, val_inputs.size(1))
sample_inputs = val_inputs[:1, :attention_token_count].to(device)

with torch.no_grad():
    _, _, attentions = model(sample_inputs, return_attentions=True)

input_token_ids = sample_inputs[0].tolist()
token_labels = format_token_labels(
    token_ids_to_labels(tokenizer, input_token_ids),
    max_tokens=attention_token_count,
    max_label_length=9,
)

single_heatmap_path = FIGURES_DIR / "attention_heatmap_layer1_head1.png"
plot_attention_heatmap(
    attention_map=attentions[0][0, 0],
    token_labels=token_labels,
    save_path=single_heatmap_path,
    title="Attention Heatmap: Layer 1, Head 1",
)

for head_idx in range(min(config.num_heads, 3)):
    save_path = FIGURES_DIR / f"attention_heatmap_layer1_head{head_idx + 1}.png"
    plot_attention_heatmap(
        attention_map=attentions[0][0, head_idx],
        token_labels=token_labels,
        save_path=save_path,
        title=f"Attention Heatmap: Layer 1, Head {head_idx + 1}",
    )
    print(f"Saved attention heatmap to: {save_path}")

for layer_idx, attention_tensor in enumerate(attentions[: min(len(attentions), 2)], start=1):
    save_path = FIGURES_DIR / f"attention_heatmap_layer{layer_idx}_head1.png"
    plot_attention_heatmap(
        attention_map=attention_tensor[0, 0],
        token_labels=token_labels,
        save_path=save_path,
        title=f"Attention Heatmap: Layer {layer_idx}, Head 1",
    )
    print(f"Saved layer comparison heatmap to: {save_path}")

heads_grid_path = FIGURES_DIR / "attention_heads_grid.png"
plot_attention_grid(
    attention_tensor=attentions[0],
    token_labels=token_labels,
    head_indices=list(range(min(config.num_heads, 4))),
    save_path=heads_grid_path,
    title="Attention Head Comparison Grid (Layer 1)",
)
print(f"Saved attention head grid to: {heads_grid_path}")

early_checkpoint = history["checkpoints"].get("epoch_1")
if early_checkpoint is not None:
    early_model = TinyTransformerLM(
        vocab_size=tokenizer.get_vocab_size(),
        context_length=config.context_length,
        d_model=config.embedding_dim,
        num_heads=config.num_heads,
        num_layers=config.num_layers,
        ff_multiplier=config.ff_multiplier,
        dropout=config.dropout,
    ).to(device)
    early_model.load_state_dict(early_checkpoint)
    early_model.eval()

    with torch.no_grad():
        _, _, early_attentions = early_model(sample_inputs, return_attentions=True)

    attention_evolution_path = FIGURES_DIR / "attention_evolution_layer1_head1.png"
    plot_attention_comparison(
        left_attention=early_attentions[0][0, 0],
        right_attention=attentions[0][0, 0],
        token_labels=token_labels,
        save_path=attention_evolution_path,
        left_title="After Epoch 1",
        right_title="Final Model",
        figure_title="Attention Evolution for Layer 1, Head 1",
    )
    print(f"Saved attention evolution comparison to: {attention_evolution_path}")

    del early_model
    if device.type == "cuda":
        torch.cuda.empty_cache()

print(f"Saved single attention heatmap to: {single_heatmap_path}")


## 19. Qualitative Generation and Lightweight Hyperparameter Comparison

Sample generations provide a quick qualitative check on whether the model learned local Shakespeare-like structure. A tiny learning-rate comparison on a small subset is also included so the report can discuss hyperparameter sensitivity without turning the notebook into a large experiment grid.


In [ ]:
@torch.no_grad()
def generate_text(
    model: torch.nn.Module,
    tokenizer: Tokenizer,
    prompt: str,
    device: torch.device,
    max_new_tokens: int = 80,
    temperature: float = 1.0,
) -> str:
    """Generate text from a prompt for a qualitative sanity check."""

    prompt_ids = tokenizer.encode(prompt).ids
    if not prompt_ids:
        raise ValueError("Prompt must produce at least one token.")

    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    output_ids = model.generate(
        input_ids, max_new_tokens=max_new_tokens, temperature=temperature
    )[0].tolist()
    return tokenizer.decode(output_ids)


prompts = ["ROMEO:", "JULIET:"]
generations: list[str] = []

for prompt in prompts:
    try:
        generated_text = generate_text(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=config.max_generation_tokens,
            temperature=0.9,
        )
        generations.append(f"Prompt: {prompt}\n{generated_text}\n")
        print(f"\nPrompt: {prompt}")
        print(generated_text)
    except RuntimeError as exc:
        print(f"Generation failed for {prompt}: {exc}")

generation_path = FIGURES_DIR / "sample_generations.txt"
generation_path.write_text("\n".join(generations), encoding="utf-8")
print(f"Saved sample generations to: {generation_path}")


def run_lightweight_learning_rate_comparison(
    learning_rates: list[float],
    subset_train_size: int = 512,
    subset_val_size: int = 128,
    epochs: int = 2,
) -> dict[str, dict[str, list[float] | dict[str, dict[str, torch.Tensor]]]]:
    """Run a very small sweep so the report can discuss hyperparameter sensitivity."""

    comparison_results = {}
    sub_train_inputs = train_inputs[: min(subset_train_size, len(train_inputs))]
    sub_train_targets = train_targets[: min(subset_train_size, len(train_targets))]
    sub_val_inputs = val_inputs[: min(subset_val_size, len(val_inputs))]
    sub_val_targets = val_targets[: min(subset_val_size, len(val_targets))]

    comparison_train_loader, comparison_val_loader = create_dataloaders(
        train_inputs=sub_train_inputs,
        train_targets=sub_train_targets,
        val_inputs=sub_val_inputs,
        val_targets=sub_val_targets,
        batch_size=min(config.batch_size, 64),
    )

    for learning_rate in learning_rates:
        seed_everything(config.seed)
        experiment_model = TinyTransformerLM(
            vocab_size=tokenizer.get_vocab_size(),
            context_length=config.context_length,
            d_model=config.embedding_dim,
            num_heads=config.num_heads,
            num_layers=config.num_layers,
            ff_multiplier=config.ff_multiplier,
            dropout=config.dropout,
        ).to(device)
        experiment_optimizer = torch.optim.AdamW(
            experiment_model.parameters(),
            lr=learning_rate,
            weight_decay=config.weight_decay,
        )

        print(f"\nRunning lightweight comparison for learning rate = {learning_rate:g}")
        experiment_history = fit(
            model=experiment_model,
            train_loader=comparison_train_loader,
            val_loader=comparison_val_loader,
            optimizer=experiment_optimizer,
            device=device,
            epochs=epochs,
        )
        comparison_results[f"lr={learning_rate:g}"] = experiment_history

        del experiment_model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    return comparison_results


comparison_results = run_lightweight_learning_rate_comparison(
    learning_rates=[1e-4, 3e-4],
    subset_train_size=512,
    subset_val_size=128,
    epochs=2,
)

learning_rate_plot_path = FIGURES_DIR / "learning_rate_comparison.png"
plot_experiment_metric_comparison(
    metric_history_by_label={
        label: result["val_loss"] for label, result in comparison_results.items()
    },
    save_path=learning_rate_plot_path,
    title="Learning Rate Comparison on a Small Subset",
    ylabel="Validation loss",
)
print(f"Saved hyperparameter comparison figure to: {learning_rate_plot_path}")

comparison_lines = ["Learning-rate comparison summary"]
for label, result in comparison_results.items():
    comparison_lines.append(
        f"{label}: final val loss = {result['val_loss'][-1]:.4f}, "
        f"final val ppl = {result['val_perplexity'][-1]:.2f}, "
        f"total runtime = {sum(result['epoch_seconds']):.2f}s"
    )

comparison_summary_path = FIGURES_DIR / "learning_rate_comparison_summary.txt"
comparison_summary_path.write_text("\n".join(comparison_lines), encoding="utf-8")
print("\n".join(comparison_lines))
print(f"Saved comparison summary to: {comparison_summary_path}")


## 20. Reflection / Analysis Section

- **Attention maps:** The heatmaps often show strong diagonal structure, which suggests tokens rely heavily on recent context, while some heads may place extra weight on repeated names, punctuation, or line-boundary patterns.
- **Training stability:** Stable decreases in both training and validation loss indicate the optimization setup is reasonable for this small model. Large divergence between the two curves would suggest overfitting or an overly aggressive learning rate.
- **Important hyperparameters:** Context length, learning rate, and hidden size have the biggest effect on runtime and stability. Larger models can fit better but quickly increase memory and compute cost.
- **Why positional encoding matters:** Without positional information, attention would treat sequences as bags of tokens and would not know whether a word appeared early or late in the context window.
- **Runtime and memory bottlenecks:** Attention becomes expensive as context length grows because the attention matrix scales with sequence length squared. Batch size and embedding dimension also directly affect memory use.

### Requirement Check

- Tiny Shakespeare corpus loaded from a local file
- Subword tokenizer used with vocabulary size at most 500
- Overlapping fixed-length token sequences created
- Input and shifted target sequences created
- 80/20 train-validation split used
- Token embedding included
- Sinusoidal positional encoding added
- Self-attention implemented manually
- Causal mask applied
- Feed-forward network implemented
- Residual connections used
- RMSNorm used
- Cross-entropy loss used
- Training and validation loss curves plotted
- Attention heatmaps plotted
- Validation perplexity computed and reported
- Implementation written in PyTorch
- Model kept lightweight by default

This notebook therefore covers the full assignment pipeline end-to-end in a single, reproducible workflow.